# Análisis exploratorio de datos

Este capítulo contiene el EDA completo en Python. Las secciones 1 a 1.10 responden al objetivo específico 1 (conocer los datos), la sección 1.11 al objetivo específico 2 (la variable objetivo del modelo) y las {ref}`conclusiones <conclusiones>` al objetivo específico 3 (implicaciones para el modelo). Ver los {ref}`objetivos <objetivos>`. Cada resultado va seguido de su **interpretación**.

```{admonition} Cómo leer este capítulo
:class: note
Las cajas verdes **Interpretación** explican qué muestra cada tabla o gráfico. Las cajas naranjas **Precaución** señalan problemas de calidad de datos que cambian la lectura de un resultado.
```

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 6), "figure.dpi": 110, "axes.titlesize": 15, "axes.titleweight": "bold", "font.size": 11, "axes.edgecolor": "#BBBBBB"})
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 160)

df = pd.read_csv("dataset_calidad_aire.csv")
print("CSV cargado, iniciando limpieza...")

In [ ]:
# Limpio columnas que vienen como strings pero son números
df["Año"] = df["Año"].astype(str).str.replace(",", "").astype(int)
columnas_numericas = ["ID Estacion", "Suma", "No. de datos", "Promedio", "Mediana", "Percentil 98", "Máximo", "Mínimo"]
for col in columnas_numericas:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(",", "").str.strip()
        df[col] = pd.to_numeric(df[col], errors="coerce")
print("Limpieza completada")
print(df.dtypes)

```{admonition} Interpretación
:class: tip
Las columnas numéricas venían como texto con comas de miles (por ejemplo `"1,003.6"`). Al quitar las comas y convertirlas, todas quedan como `float64` o `int64` y se pueden usar en cálculos. Sin este paso, cualquier promedio o gráfico habría fallado o devuelto valores nulos.
```

In [ ]:
# Helper para mostrar tablas parecidas a las de knitr::kable + kableExtra
def tabla(d, columnas=None, decimales=2):
    d = d.copy()
    if columnas is not None: d.columns = columnas
    return d.round(decimales).style.hide(axis="index").set_table_styles([{"selector": "th, td", "props": [("border", "1px solid #999999"), ("padding", "4px 10px"), ("text-align", "left")]}])

## 1. Preparación y calidad de los datos

*Objetivo específico 1 · calidad de los datos.*

In [ ]:
df.shape

```{admonition} Interpretación
:class: tip
El archivo original tiene 29.683 filas y 28 columnas. Cada fila es el **resumen anual** de una variable, en una estación y para un tiempo de exposición, no una medición individual.
```

In [ ]:
df.info()

In [ ]:
list(df.columns)

In [ ]:
df.isna().sum()

In [ ]:
porcentaje_na = df.isna().sum() / len(df) * 100
porcentaje_na

```{admonition} Interpretación
:class: tip
Solo tres columnas tienen faltantes y ninguna supera el 0,5 %: `Representatividad Temporal` (148 filas), `Código del Municipio` (4) y `Tipo de Estación` (2). Las columnas que usa el análisis (`Variable`, `Año`, `Promedio`) están completas, así que no hace falta imputar ni eliminar filas por faltantes.
```

In [ ]:
df.duplicated().sum()

In [ ]:
df_unicos = df.drop_duplicates()
print("Filas originales:", len(df))
print("Filas únicas:", len(df_unicos))
print("Duplicados:", len(df) - len(df_unicos))

In [ ]:
porcentaje_duplicados = df.duplicated().sum() / len(df) * 100
print("Porcentaje de filas duplicadas:", round(porcentaje_duplicados, 2), "%")

```{admonition} Interpretación
:class: tip
Casi la mitad del archivo (46,96 %) son filas idénticas a otra. Una proporción así indica que el consolidado se armó juntando reportes que se solaparon, no que se repitieran mediciones reales. Si no se eliminaran, cada estación duplicada pesaría el doble en promedios y conteos. Por eso el resto del análisis trabaja con las **15.745 filas únicas**.
```

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape

In [ ]:
df.duplicated().sum()

### 1.1 Exploración general del dataset

*Objetivo específico 1 · cobertura del monitoreo.*

In [ ]:
df["Variable"].value_counts().sort_index()

In [ ]:
df["Variable"].nunique()

```{admonition} Interpretación
:class: tip
Hay 20 variables, que se agrupan en contaminantes (PM10, PM2.5, PST, NO, NO₂, O₃, SO₂, CO) y meteorológicas (temperatura, humedad, viento, presión, precipitación y radiación). La temperatura y la humedad aparecen en tres versiones (sin altura, a 2 m y a 10 m), que son variables distintas y no deben mezclarse.
```

#### 1.1.1 Distribución de registros por estación

In [ ]:
df["Estación"].nunique()

In [ ]:
registros_estacion = df["Estación"].value_counts().rename_axis("Estación").reset_index(name="n")
registros_estacion.head(10)

In [ ]:
def barras_h(d, cat, val, titulo, xlab, ylab, subtitulo=None, alto=6, tam=9, dec=0):
    d = d.sort_values(val)
    fig, ax = plt.subplots(figsize=(9, alto))
    ax.barh(d[cat].astype(str), d[val], color="#595959")
    for y, v in zip(d[cat].astype(str), d[val]): ax.text(v * 1.01, y, f"{v:.{dec}f}", va="center", fontsize=tam)
    ax.set_xlim(0, d[val].max() * 1.08)
    ax.set_title(titulo if subtitulo is None else titulo + "\n" + subtitulo, fontweight="bold")
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()

In [ ]:
top_estaciones = registros_estacion.nlargest(15, "n")
barras_h(top_estaciones, "Estación", "n", "15 estaciones con mayor número de registros", "Número de registros", "Estación")

```{admonition} Interpretación
:class: tip
Las estaciones con más registros (Centro de Alto Rendimiento, Kennedy, Tunal, Las Ferias, Guaymaral, Puente Aranda) son todas de **Bogotá**. Son estaciones que miden muchas variables a la vez durante todo el periodo, por eso dominan el conteo.
```

#### 1.1.2 Distribución de registros por variable ambiental

In [ ]:
frecuencia_variables = df["Variable"].value_counts().rename_axis("Variable").reset_index(name="n")
frecuencia_variables

In [ ]:
barras_h(frecuencia_variables, "Variable", "n", "Número de registros por variable ambiental", "Número de registros", "Variable ambiental", alto=7)

```{admonition} Interpretación
:class: tip
PM10 y PM2.5 suman casi un tercio de los registros: el material particulado es el contaminante más vigilado por su efecto en la salud. En el otro extremo, TAire, HAire y RUVb tienen menos de 80 registros cada uno, así que cualquier conclusión sobre ellos descansa en muy poca información.
```

### 1.2 Distribución temporal de los registros

In [ ]:
frecuencia_anios = df["Año"].value_counts().rename_axis("Año").reset_index(name="n").sort_values("Año")
frecuencia_anios

In [ ]:
fig, ax = plt.subplots()
ax.bar(frecuencia_anios["Año"].astype(str), frecuencia_anios["n"], color="#595959")
for x, v in zip(frecuencia_anios["Año"].astype(str), frecuencia_anios["n"]): ax.text(x, v * 1.01, str(v), ha="center", fontsize=9)
ax.set_ylim(0, frecuencia_anios["n"].max() * 1.08)
ax.set_title("Número de registros por año", fontweight="bold"); ax.set_xlabel("Año"); ax.set_ylabel("Número de registros")
plt.xticks(rotation=45, ha="right"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Los registros pasan de 647 en 2011 a 1.941 en 2022: la red de monitoreo se triplicó. La caída de 2023–2024 puede deberse a reportes incompletos o al cierre de estaciones; el dataset no permite distinguirlo.

**Consecuencia para el análisis:** cuando el promedio anual de una variable cambia, parte del cambio puede venir de que entraron o salieron estaciones, y no solo de cambios reales en el aire.
```

### 1.3 Estadística descriptiva y distribución de variables ambientales

In [ ]:
estadistica_variables = df.groupby("Variable", as_index=False).agg(n=("Promedio", "size"), Media=("Promedio", "mean"), Mediana=("Promedio", "median"), Desviacion_Estandar=("Promedio", "std"), Minimo=("Promedio", "min"), Q1=("Promedio", lambda s: s.quantile(0.25)), Q3=("Promedio", lambda s: s.quantile(0.75)), Maximo=("Promedio", "max"))
estadistica_variables

```{admonition} Interpretación
:class: tip
Las escalas son muy distintas (presión cerca de 700 mmHg, viento cerca de 1 m/s), así que cada variable solo se compara consigo misma.

En casi todos los contaminantes la media es mayor que la mediana (PM10: 37,1 frente a 34,6; SO₂: 9,4 frente a 6,0). Eso indica **asimetría positiva**: unas pocas estaciones con valores altos jalan el promedio hacia arriba.

**PLiquida** es el caso extremo: su media (9,8 mm) es siete veces la mediana (1,4 mm) por unos pocos valores enormes (máximo 3.236,7 mm). Para esta variable la mediana es la medida de tendencia central adecuada.
```

#### Distribución de las variables ambientales

In [ ]:
vars_orden = sorted(df["Variable"].unique())
def panel_boxplots(inicio, fin, subtitulo):
    grupo = vars_orden[inicio:fin]
    filas = int(np.ceil(len(grupo) / 2))
    fig, axes = plt.subplots(filas, 2, figsize=(9, 3.2 * filas))
    axes = np.array(axes).ravel()
    for ax, v in zip(axes, grupo):
        ax.boxplot(df.loc[df["Variable"] == v, "Promedio"].dropna(), widths=0.5)
        ax.set_title(v, fontweight="bold", fontsize=12); ax.set_xticks([]); ax.set_ylabel("Valor promedio"); ax.grid(axis="x", visible=False)
    for ax in axes[len(grupo):]: ax.axis("off")
    fig.suptitle("Distribución de las variables ambientales\n" + subtitulo, fontweight="bold", fontsize=15)
    plt.tight_layout(); plt.show()

In [ ]:
panel_boxplots(0, 5, "Variables ambientales 1-5")

In [ ]:
panel_boxplots(5, 10, "Variables ambientales 6-10")

In [ ]:
panel_boxplots(10, 15, "Variables ambientales 11-15")

In [ ]:
panel_boxplots(15, 20, "Variables ambientales 16-20")

```{admonition} Interpretación
:class: tip
Los diagramas de caja confirman la asimetría: en la mayoría de las variables los puntos aislados aparecen por encima de la caja, no por debajo. RUVb tiene una caja enorme porque mezcla dos escalas de medición distintas (ver sección 1.7).
```

#### Identificación de valores atípicos

In [ ]:
outliers_variables = df.groupby("Variable", as_index=False).agg(Q1=("Promedio", lambda s: s.quantile(0.25)), Q3=("Promedio", lambda s: s.quantile(0.75)))
outliers_variables["IQR"] = outliers_variables["Q3"] - outliers_variables["Q1"]
outliers_variables["Limite_inferior"] = outliers_variables["Q1"] - 1.5 * outliers_variables["IQR"]
outliers_variables["Limite_superior"] = outliers_variables["Q3"] + 1.5 * outliers_variables["IQR"]

#### Resumen de valores atípicos

In [ ]:
tabla(outliers_variables)

```{admonition} Interpretación
:class: tip
Cuando el límite inferior es negativo (CO, NO, NO₂, SO₂, PLiquida) no puede haber atípicos bajos, porque esas variables no toman valores negativos. En esos casos los atípicos son solo valores extremadamente altos.
```

Uso el criterio de Tukey: todo lo que quede por fuera de Q1 menos 1.5 veces el IQR y Q3 más 1.5 veces el IQR lo considero atípico. Es el mismo criterio que apliqué en R.

### 1.4 Representatividad temporal de los datos

In [ ]:
representatividad_variables = df.groupby("Variable", as_index=False).agg(Media=("Representatividad Temporal", "mean"), Mediana=("Representatividad Temporal", "median"), Minimo=("Representatividad Temporal", "min"), Maximo=("Representatividad Temporal", "max")).sort_values("Media", ascending=False)
tabla(representatividad_variables)

In [ ]:
barras_h(representatividad_variables, "Variable", "Media", "Representatividad temporal promedio por variable", "Representatividad temporal (%)", "Variable ambiental", alto=7, dec=1)

```{admonition} Interpretación
:class: tip
Casi todas las variables promedian entre 63 % y 75 % de representatividad: en un registro típico falta cerca de un tercio del año. **TAire y HAire** (sin altura) son la excepción, con 24 % y 27 %: sus promedios anuales se basan en pocos meses.

Solo el 56 % de los registros alcanza el 75 % de cobertura, un umbral comúnmente usado para considerar válido un promedio anual.
```

In [ ]:
fuera_de_rango = df[df["Representatividad Temporal"] > 100]
print("Registros con representatividad > 100 %:", len(fuera_de_rango))
print("Máximo reportado:", df["Representatividad Temporal"].max())

```{admonition} Precaución
:class: warning
Una representatividad mayor a 100 % es imposible (no se puede cubrir más que el año completo). Son errores de captura en la fuente; se documentan, pero no se corrigen porque no hay forma de saber el valor real.
```

### 1.5 Distribución geográfica de las estaciones

In [ ]:
ubicacion_estaciones = df[["Estación", "Latitud", "Longitud"]].drop_duplicates().sort_values(["Latitud", "Longitud"])
tabla(ubicacion_estaciones.head(20), ["Estación", "Latitud", "Longitud"], decimales=4)

```{admonition} Interpretación
:class: tip
Las estaciones están en todo el país, desde Pasto (latitud 1,2°) hacia el norte. Algunas aparecen repetidas con coordenadas casi idénticas o con un asterisco en el nombre (`VIVERO LA FLORIDA*`): son la misma estación reportada de dos formas.
```

#### Distribución por tipo de estación

In [ ]:
frecuencia_tipo_estacion = df["Tipo de Estación"].value_counts().rename_axis("Tipo de Estación").reset_index(name="n")
tabla(frecuencia_tipo_estacion, ["Tipo de estación", "Número de registros"])

In [ ]:
barras_h(frecuencia_tipo_estacion, "Tipo de Estación", "n", "Distribución de registros por tipo de estación", "Número de registros", "Tipo de estación", alto=5)

```{admonition} Interpretación
:class: tip
El 90 % de los registros viene de **estaciones fijas**, que operan de forma continua. Las indicativas son campañas temporales con menos cobertura en el año.
```

#### Distribución por departamento

In [ ]:
frecuencia_departamentos = df["Nombre del Departamento"].value_counts().rename_axis("Nombre del Departamento").reset_index(name="n")
tabla(frecuencia_departamentos, ["Departamento", "Número de registros"])

In [ ]:
departamentos_1 = frecuencia_departamentos.iloc[0:24]
barras_h(departamentos_1, "Nombre del Departamento", "n", "Registros por departamento", "Número de registros", "Departamento", subtitulo="Departamentos 1-24", alto=8, tam=8)

In [ ]:
departamentos_2 = frecuencia_departamentos.iloc[24:49]
barras_h(departamentos_2, "Nombre del Departamento", "n", "Registros por departamento", "Número de registros", "Departamento", subtitulo="Departamentos 25-49", alto=8, tam=8)

```{admonition} Precaución
:class: warning
En la tabla anterior el mismo departamento aparece escrito de varias formas: `BOGOTÁ, D.C.`, `BOGOTA, D.C.` y `Bogotá, D.C.`; `ANTIOQUIA` y `Antioquia`, etc. Por eso el conteo da 49 "departamentos". Abajo se normalizan los nombres (sin tildes y en mayúsculas) para obtener el número real.
```

In [ ]:
import unicodedata
def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode()
    return " ".join(t.upper().split())

df["Departamento_norm"] = df["Nombre del Departamento"].map(normalizar)
df["Municipio_norm"] = df["Nombre del Municipio"].map(normalizar)
print("Departamentos: ", df["Nombre del Departamento"].nunique(), "nombres ->", df["Departamento_norm"].nunique(), "reales")
print("Municipios:    ", df["Nombre del Municipio"].nunique(), "nombres ->", df["Municipio_norm"].nunique(), "reales")
deptos_norm = df["Departamento_norm"].value_counts().rename_axis("Departamento").reset_index(name="n")
barras_h(deptos_norm, "Departamento", "n", "Registros por departamento", "Número de registros", "Departamento",
         subtitulo="Nombres normalizados", alto=7, tam=8)

```{admonition} Interpretación
:class: tip
Al normalizar quedan **24 departamentos** (no 49) y **193 municipios** (no 339). Antioquia y Bogotá concentran la mitad de los registros, seguidos por Valle del Cauca y Cundinamarca. La Amazonía, la Orinoquía y gran parte del Pacífico casi no tienen monitoreo: los resultados nacionales representan sobre todo a las ciudades andinas.
```

#### Distribución por municipio

In [ ]:
frecuencia_municipios = df["Nombre del Municipio"].value_counts().rename_axis("Nombre del Municipio").reset_index(name="n")
tabla(frecuencia_municipios, ["Municipio", "Número de registros"])

In [ ]:
top_municipios = frecuencia_municipios.nlargest(24, "n")
barras_h(top_municipios, "Nombre del Municipio", "n", "Municipios con mayor número de registros", "Número de registros", "Municipio", subtitulo="24 municipios con mayor representación en la base de datos", alto=8, tam=8)

```{admonition} Interpretación
:class: tip
Los municipios con más registros son las capitales con redes de monitoreo grandes: Bogotá, Cali, Medellín y Bucaramanga. Aparecen también municipios industriales o mineros como Sogamoso, Nobsa (cementeras y siderurgia) y La Jagua de Ibirico (carbón), donde la vigilancia responde a fuentes de emisión específicas.
```

### 1.6 Análisis específico de contaminantes

*Objetivo específico 1 · evolución de los contaminantes.*

In [ ]:
def estadisticos(serie, con_media=True):
    nombres = ["Mínimo", "Primer cuartil", "Mediana", "Media", "Tercer cuartil", "Máximo"]
    valores = [serie.min(), serie.quantile(0.25), serie.median(), serie.mean(), serie.quantile(0.75), serie.max()]
    if not con_media: nombres, valores = ["Mínimo", "Primer cuartil", "Mediana", "Máximo"], [serie.min(), serie.quantile(0.25), serie.median(), serie.max()]
    return pd.DataFrame({"Estadistico": nombres, "Valor": valores})

def promedio_anual(d, nombre): return d.groupby("Año", as_index=False).agg(**{nombre: ("Promedio", "mean"), "Registros": ("Promedio", "size")}).sort_values("Año").reset_index(drop=True)

def grafico_anual(d, col, titulo, subtitulo, ylab, anios=None):
    fig, ax = plt.subplots()
    ax.plot(d["Año"], d[col], linewidth=1.2, marker="o", markersize=6, color="black")
    for x, y in zip(d["Año"], d[col]): ax.text(x, y + (d[col].max() - d[col].min() + 1) * 0.03, f"{y:.1f}", ha="center", fontsize=9)
    ax.set_xticks(list(anios) if anios is not None else sorted(d["Año"].unique()))
    ax.set_ylim(top=d[col].max() * 1.08)
    ax.set_title(titulo + "\n" + subtitulo, fontweight="bold"); ax.set_xlabel("Año"); ax.set_ylabel(ylab)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()

def cambio_porcentual(d, col, a1=2011, a2=2024):
    s = d.set_index("Año")[col]
    if a1 not in s.index or a2 not in s.index: return pd.DataFrame({"Periodo": ["N/A"], "Promedio_2011": [np.nan], "Promedio_2024": [np.nan], "Cambio_porcentual": [np.nan]})
    v1, v2 = s.loc[a1], s.loc[a2]
    return pd.DataFrame({"Periodo": [f"{a1}-{a2}"], "Promedio_2011": [v1], "Promedio_2024": [v2], "Cambio_porcentual": [(v2 - v1) / v1 * 100]})

#### PM10

In [ ]:
pm10 = df[df["Variable"] == "PM10"].copy()
print("Registros de PM10:", len(pm10))
estadistica_pm10 = estadisticos(pm10["Promedio"])
tabla(estadistica_pm10, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
PM10 tiene 2.984 registros. La media (37,1 µg/m³) es algo mayor que la mediana (34,6 µg/m³), lo que indica asimetría positiva: algunas estaciones con valores altos, como el máximo de 162,3 µg/m³, elevan el promedio.
```

In [ ]:
pm10_anual = promedio_anual(pm10, "Promedio_PM10")
tabla(pm10_anual, ["Año", "Promedio PM10", "Número de registros"])

In [ ]:
grafico_anual(pm10_anual, "Promedio_PM10", "Evolución del promedio anual de PM10", "Periodo 2011-2024", "Promedio de PM10", anios=range(2011, 2025))

```{admonition} Interpretación
:class: tip
El PM10 se mantuvo cerca de 43–44 µg/m³ entre 2011 y 2016 y desde 2017 baja de forma sostenida hasta 30,8 µg/m³ en 2024. Es la tendencia más estable de todos los contaminantes.

Como referencia, la Resolución 2254 de 2017 fija un nivel anual de 50 µg/m³ y una meta de 30 µg/m³ a 2030: el promedio nacional estuvo por debajo del nivel vigente todos los años y en 2024 quedó muy cerca de la meta futura. Es solo una referencia, porque el promedio mezcla registros de 1 y 24 horas de exposición.
```

In [ ]:
tabla_cambio_pm10 = cambio_porcentual(pm10_anual, "Promedio_PM10")
tabla(tabla_cambio_pm10, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el PM10 bajó **27,75 %**.
```

#### PM2.5

In [ ]:
pm25 = df[df["Variable"] == "PM2.5"].copy()
print("Registros de PM2.5:", len(pm25))
estadistica_pm25 = estadisticos(pm25["Promedio"])
tabla(estadistica_pm25, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El PM2.5 tiene menos registros (1.984) y valores más bajos que el PM10, lo esperable porque es la fracción fina del material particulado. La media (16,8 µg/m³) y la mediana (15,7 µg/m³) son cercanas.
```

In [ ]:
pm25_anual = promedio_anual(pm25, "Promedio_PM25")
tabla(pm25_anual, ["Año", "Promedio PM2.5", "Número de registros"])

In [ ]:
grafico_anual(pm25_anual, "Promedio_PM25", "Evolución del promedio anual de PM2.5", "Periodo 2011-2024", "Promedio de PM2.5", anios=range(2011, 2025))

```{admonition} Interpretación
:class: tip
El PM2.5 cae con fuerza en 2017, de 23,7 a 18,6 µg/m³. Ese salto coincide con que el número de registros se duplicó (de 65 a 142): al entrar estaciones nuevas, posiblemente en zonas menos contaminadas, el promedio puede bajar sin que el aire haya mejorado en las estaciones existentes. Es un **efecto de composición de la red** que este EDA no puede descartar.

Desde 2023 el promedio nacional (14,0–14,7 µg/m³) está por debajo de la meta a 2030 de la Res. 2254/2017 (15 µg/m³).
```

In [ ]:
tabla_cambio_pm25 = cambio_porcentual(pm25_anual, "Promedio_PM25")
tabla(tabla_cambio_pm25, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el PM2.5 bajó **31,35 %**.
```

#### NO₂

In [ ]:
no2 = df[df["Variable"] == "NO2"].copy()
print("Registros de NO2:", len(no2))
estadistica_no2 = estadisticos(no2["Promedio"])
tabla(estadistica_no2, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El NO₂ tiene solo 644 registros: entre 20 y 59 por año. La media (26,5 µg/m³) y la mediana (26,0 µg/m³) son prácticamente iguales, una distribución bastante simétrica.
```

In [ ]:
no2_anual = promedio_anual(no2, "Promedio_NO2")
tabla(no2_anual, ["Año", "Promedio NO2", "Número de registros"])

In [ ]:
grafico_anual(no2_anual, "Promedio_NO2", "Evolución del promedio anual de NO2", "Periodo 2011-2024", "Promedio de NO2", anios=range(2011, 2025))

```{admonition} Interpretación
:class: tip
El NO₂ bajó con fuerza entre 2011 y 2017 (de 36,3 a 21,3 µg/m³), pero desde entonces sube lentamente hasta 26,9 µg/m³ en 2024. El cambio total esconde ese repunte. El NO₂ proviene sobre todo de la combustión vehicular, así que el repunte podría reflejar el aumento del parque automotor. Todos los valores están muy por debajo del nivel anual vigente (60 µg/m³) y de la meta 2030 (40 µg/m³).
```

In [ ]:
tabla_cambio_no2 = cambio_porcentual(no2_anual, "Promedio_NO2")
tabla(tabla_cambio_no2, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el NO₂ bajó **25,9 %**, pero con el repunte desde 2017 descrito arriba.
```

#### O₃

In [ ]:
o3 = df[df["Variable"] == "O3"].copy()
print("Registros de O3:", len(o3))
estadistica_o3 = estadisticos(o3["Promedio"])
tabla(estadistica_o3, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El ozono tiene 827 registros, con media y mediana cercanas (26,2 y 25,7 µg/m³).
```

In [ ]:
o3_anual = promedio_anual(o3, "Promedio_O3")
tabla(o3_anual, ["Año", "Promedio O3", "Número de registros"])

In [ ]:
grafico_anual(o3_anual, "Promedio_O3", "Evolución del promedio anual de O3", "Periodo 2011-2024", "Promedio de O3", anios=range(2011, 2025))

```{admonition} Interpretación
:class: tip
El ozono no muestra una tendencia clara: sube hasta 32,6 µg/m³ en 2015, baja hasta 22,7 µg/m³ en 2021 y vuelve a subir en 2024. El O₃ no se emite directamente: se forma en la atmósfera a partir de óxidos de nitrógeno y compuestos orgánicos con la radiación solar. Por eso puede subir aunque bajen otros contaminantes.
```

In [ ]:
tabla_cambio_o3 = cambio_porcentual(o3_anual, "Promedio_O3")
tabla(tabla_cambio_o3, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
El ozono es el único contaminante que **aumentó** entre 2011 y 2024 (+14 %).
```

#### SO₂

In [ ]:
so2 = df[df["Variable"] == "SO2"].copy()
print("Registros de SO2:", len(so2))
estadistica_so2 = estadisticos(so2["Promedio"])
tabla(estadistica_so2, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El SO₂ es muy asimétrico: la media (9,4 µg/m³) es mucho mayor que la mediana (6,0 µg/m³) y el máximo llega a 91,7 µg/m³. Unas pocas estaciones cercanas a fuentes industriales concentran los valores altos.
```

In [ ]:
so2_anual = promedio_anual(so2, "Promedio_SO2")
tabla(so2_anual, ["Año", "Promedio SO2", "Número de registros"])

In [ ]:
grafico_anual(so2_anual, "Promedio_SO2", "Evolución del promedio anual de SO2", "Periodo 2011-2024", "Promedio de SO2", anios=range(2011, 2025))

```{admonition} Interpretación
:class: tip
La serie del SO₂ es muy irregular: baja a 5,9 µg/m³ en 2014 y sube a 15,9 µg/m³ en 2016. Ese comportamiento de sierra sugiere que el promedio depende de pocas estaciones industriales que entran y salen de la red.
```

In [ ]:
tabla_cambio_so2 = cambio_porcentual(so2_anual, "Promedio_SO2")
tabla(tabla_cambio_so2, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
La caída de **34,8 %** es la mayor de los cinco contaminantes, pero por la irregularidad de la serie no debe leerse como una mejora clara y sostenida.
```

#### Comparación del cambio porcentual de los contaminantes

In [ ]:
cambios = []
for nombre, dat in [("PM10", pm10_anual), ("PM2.5", pm25_anual), ("NO2", no2_anual), ("O3", o3_anual), ("SO2", so2_anual)]:
    s = dat.set_index("Año").iloc[:, 0]
    if 2011 in s.index and 2024 in s.index:
        v1, v2 = s.loc[2011], s.loc[2024]
        cambios.append({"Contaminante": nombre, "Promedio_2011": v1, "Promedio_2024": v2, "Cambio_porcentual": (v2 - v1) / v1 * 100})
comparacion_contaminantes = pd.DataFrame(cambios)
tabla(comparacion_contaminantes, ["Contaminante", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

In [ ]:
d = comparacion_contaminantes.sort_values("Cambio_porcentual")
fig, ax = plt.subplots()
ax.barh(d["Contaminante"], d["Cambio_porcentual"], color="#595959")
for y, v in zip(d["Contaminante"], d["Cambio_porcentual"]): ax.text(v * 1.03, y, f"{v:.1f}%", va="center", ha="right" if v < 0 else "left", fontsize=9)
ax.set_xlim(d["Cambio_porcentual"].min() * 1.15, d["Cambio_porcentual"].max() * 1.15)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Cambio porcentual de los principales contaminantes\nVariación del promedio entre 2011 y 2024", fontweight="bold")
ax.set_xlabel("Cambio porcentual (%)"); ax.set_ylabel("Contaminante"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Cuatro de los cinco contaminantes bajaron entre 2011 y 2024, y el material particulado, el más relevante para la salud, se redujo cerca de 30 %. El ozono es la excepción.

El tamaño del cambio no indica qué tan confiable es: el SO₂ tiene la mayor caída pero la serie más irregular, y el PM10 una caída menor pero la tendencia más estable.
```

### 1.7 Análisis de variables meteorológicas

*Objetivo específico 1 · variables meteorológicas.*

#### Temperatura del aire (TAire)

In [ ]:
taire = df[df["Variable"] == "TAire"].copy()
print("Registros de TAire:", len(taire))
estadistica_taire = estadisticos(taire["Promedio"])
tabla(estadistica_taire, ["Estadístico", "Valor"])

In [ ]:
taire_anual = promedio_anual(taire, "Promedio_TAire")
tabla(taire_anual, ["Año", "Promedio TAire", "Número de registros"])

```{admonition} Interpretación
:class: tip
TAire solo tiene datos en 6 años (2011–2013 y 2017–2019) y 73 registros; algunos años tienen apenas 2.
```

In [ ]:
grafico_anual(taire_anual, "Promedio_TAire", "Evolución del promedio anual de temperatura del aire", "Años con registros disponibles", "Promedio de temperatura del aire (°C)")

```{admonition} Interpretación
:class: tip
La subida de 20,9 °C a 24,9 °C no es un cambio climático plausible en 8 años: refleja que cambiaron las estaciones que reportan (una estación en tierra caliente sube el promedio). TAire2 y TAire10, con 834 y 334 registros, son mejores variables para estudiar la temperatura.
```

#### Humedad relativa del aire (HAire)

In [ ]:
haire = df[df["Variable"] == "HAire"].copy()
print("Registros de HAire:", len(haire))
estadistica_haire = estadisticos(haire["Promedio"])
tabla(estadistica_haire, ["Estadístico", "Valor"])

In [ ]:
haire_anual = promedio_anual(haire, "Promedio_HAire")
tabla(haire_anual, ["Año", "Promedio HAire", "Número de registros"])

In [ ]:
grafico_anual(haire_anual, "Promedio_HAire", "Evolución del promedio anual de humedad del aire", "Años con registros disponibles", "Promedio de humedad del aire")

```{admonition} Interpretación
:class: tip
La humedad relativa oscila entre 62 % y 73 % sin una tendencia clara. Con solo 7 años y 75 registros, y 27 % de representatividad media, sus promedios anuales son poco confiables.
```

#### Velocidad del viento (VViento)

In [ ]:
vviento = df[df["Variable"] == "VViento"].copy()
print("Registros de VViento:", len(vviento))
estadistica_vviento = estadisticos(vviento["Promedio"])
tabla(estadistica_vviento, ["Estadístico", "Valor"])

In [ ]:
vviento_anual = promedio_anual(vviento, "Promedio_VViento")
tabla(vviento_anual, ["Año", "Promedio VViento", "Número de registros"])

In [ ]:
grafico_anual(vviento_anual, "Promedio_VViento", "Evolución del promedio anual de velocidad del viento", "Años con registros disponibles", "Velocidad del viento (m/s)")

```{admonition} Interpretación
:class: tip
La velocidad del viento es baja en todo el periodo (1,2–1,7 m/s), en el rango de vientos débiles, y desde 2013 baja levemente. Con poco viento los contaminantes se dispersan menos, sobre todo en valles cerrados como el Aburrá, donde está buena parte de las estaciones.
```

#### Presión atmosférica (P)

In [ ]:
presion = df[df["Variable"] == "P"].copy()
print("Registros de P:", len(presion))
estadistica_presion = estadisticos(presion["Promedio"])
tabla(estadistica_presion, ["Estadístico", "Valor"])

In [ ]:
presion_anual = promedio_anual(presion, "Promedio_P")
tabla(presion_anual, ["Año", "Promedio de presión", "Número de registros"])

```{admonition} Interpretación
:class: tip
La presión salta de unos 635 mmHg (2011–2014) a 851 mmHg en 2018. La presión a nivel del mar es 760 mmHg y en Bogotá cerca de 560 mmHg, así que un promedio superior a 800 mmHg es **físicamente imposible**. La celda siguiente revela la causa.
```

In [ ]:
grafico_anual(presion_anual, "Promedio_P", "Evolución del promedio anual de presión atmosférica", "Años con registros disponibles", "Presión atmosférica (hPa)")

In [ ]:
sospechosos = presion[presion["Promedio"] > 800]
print("Registros de presión > 800 mmHg:", len(sospechosos), "de", len(presion))
sospechosos["Nombre del Municipio"].value_counts().head(6)

```{admonition} Precaución
:class: warning
Hay registros de más de 800 mmHg en Cali y Medellín, ciudades a 1.000–1.500 m de altitud, donde la presión real ronda 640–680 mmHg. Los valores cercanos a 1.010 corresponden a **hectopascales (hPa)** etiquetados como mmHg. Es un error de unidades: esta variable no se puede interpretar ni correlacionar sin corregirlo.
```

#### Precipitación líquida (PLiquida)

In [ ]:
pliquida = df[df["Variable"] == "PLiquida"].copy()
print("Registros de PLiquida:", len(pliquida))
estadistica_pliquida = estadisticos(pliquida["Promedio"])
tabla(estadistica_pliquida, ["Estadístico", "Valor"])

In [ ]:
pliquida_anual = promedio_anual(pliquida, "Promedio_PLiquida")
tabla(pliquida_anual, ["Año", "Promedio de precipitación", "Número de registros"])

In [ ]:
grafico_anual(pliquida_anual, "Promedio_PLiquida", "Evolución del promedio anual de precipitación líquida", "Años con registros disponibles", "Precipitación líquida (mm)")

```{admonition} Interpretación
:class: tip
La precipitación promedio está entre 2 y 7 mm en casi todos los años, pero 2017 (56,5 mm) y 2019 (34,8 mm) se disparan por unos pocos valores extremos. Probablemente son acumulados reportados como promedios. Para esta variable, la mediana es más representativa que la media.
```

#### Radiación global (RGlobal)

In [ ]:
rglobal = df[df["Variable"] == "RGlobal"].copy()
print("Registros de RGlobal:", len(rglobal))
estadistica_rglobal = estadisticos(rglobal["Promedio"])
tabla(estadistica_rglobal, ["Estadístico", "Valor"])

In [ ]:
rglobal_anual = promedio_anual(rglobal, "Promedio_RGlobal")
tabla(rglobal_anual, ["Año", "Promedio RGlobal", "Número de registros"])

In [ ]:
grafico_anual(rglobal_anual, "Promedio_RGlobal", "Evolución del promedio anual de radiación global", "Años con registros disponibles", "Radiación global (W/m²)")

```{admonition} Interpretación
:class: tip
La radiación global es estable, entre 165 y 199 W/m², con un leve aumento hasta 2020. Es coherente con la ubicación ecuatorial del país, donde la radiación varía poco a lo largo del año.
```

#### Radiación ultravioleta B (RUVb)

In [ ]:
ruvb = df[df["Variable"] == "RUVb"].copy()
print("Registros de RUVb:", len(ruvb))
estadistica_ruvb = estadisticos(ruvb["Promedio"])
tabla(estadistica_ruvb, ["Estadístico", "Valor"])

In [ ]:
ruvb_anual = promedio_anual(ruvb, "Promedio_RUVb")
tabla(ruvb_anual, ["Año", "Promedio RUVb", "Número de registros"])

In [ ]:
grafico_anual(ruvb_anual, "Promedio_RUVb", "Evolución del promedio anual de radiación UV-B", "Años con registros disponibles", "Radiación UV-B")

```{admonition} Interpretación
:class: tip
La radiación UV-B pasa de 0,6 (2011–2014) a valores entre 130 y 180 desde 2019. Un cambio así no es atmosférico; la celda siguiente muestra la causa.
```

In [ ]:
ruvb.groupby("Año")["Autoridad Ambiental"].unique()

```{admonition} Precaución
:class: warning
Hasta 2014 la RUVb solo la reportaba la SDA (Bogotá); desde 2017 la reportan otras autoridades (CDMB, Corpoboyacá, CRC) con otra escala. La serie **no es comparable** entre los dos periodos, y cualquier correlación con RUVb será un artefacto de este cambio.
```

### 1.8 Análisis de correlaciones

*Objetivo específico 1 · asociaciones entre variables.*

#### Preparación de los datos

In [ ]:
datos_correlacion = df.groupby(["Año", "Variable"], as_index=False)["Promedio"].mean().pivot(index="Año", columns="Variable", values="Promedio").reset_index().rename_axis(None, axis=1)
datos_correlacion.head()

```{admonition} Interpretación
:class: tip
Para correlacionar variables que se miden en estaciones distintas, se calcula el promedio anual nacional de cada una. Cada variable queda reducida a **máximo 14 puntos** (uno por año), y algunas tienen menos (TAire 6, HAire 7). Con tan pocos puntos, un solo año atípico puede cambiar mucho el coeficiente.
```

#### Matriz de correlaciones

In [ ]:
variables_eda = ["PM10", "PM2.5", "NO2", "SO2", "O3", "TAire", "HAire", "VViento", "P", "PLiquida", "RGlobal", "RUVb"]
matriz_correlacion = datos_correlacion[[v for v in variables_eda if v in datos_correlacion.columns]].corr(method="pearson")
matriz_correlacion.round(2)

In [ ]:
distancia = 1 - matriz_correlacion
orden = leaves_list(linkage(squareform(distancia.values, checks=False), method="complete"))
m = matriz_correlacion.iloc[orden, orden]
mascara = np.tril(np.ones(m.shape, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(m, mask=mascara, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap="RdBu_r", vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Matriz de correlación de Pearson", fontweight="bold")
plt.xticks(rotation=45, ha="left"); ax.xaxis.tick_top()
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
El agrupamiento jerárquico forma dos bloques. El primero reúne los contaminantes (PM10, PM2.5, O₃, SO₂, NO₂) con la velocidad del viento y la humedad, que se correlacionan positivamente entre sí. El segundo agrupa temperatura, radiación, presión y precipitación, que en general se relacionan negativamente con los contaminantes.

- **Relación robusta:** PM10–PM2.5 (r = 0,97, 14 años). El PM2.5 es una fracción del PM10 y ambos comparten fuentes (tráfico, industria, polvo resuspendido). Es la única relación fuerte sustentada por la serie completa y sin problemas de datos.
- **Plausibles pero frágiles:** TAire con PM10 (−0,79) y con NO₂ (−0,73). Más temperatura favorece la mezcla vertical y la dispersión, pero estos valores se basan en solo 6 años y en una variable con 24 % de representatividad.
- **Espurias:** todas las de RUVb (con PM10 −0,83, con TAire 0,85) y las de P, por el cambio de escala y el error de unidades vistos en la sección 1.7.
- **Con interés físico:** NO₂–RGlobal (−0,71, 12 años). La radiación solar descompone el NO₂ (fotólisis) y genera ozono, lo que es coherente con la relación negativa.

Una correlación alta no implica causalidad.
```

#### Tabla de correlaciones y fuerza de asociación

In [ ]:
def fuerza(r):
    a = abs(r)
    if a >= 0.80: return "Muy fuerte"
    elif a >= 0.60: return "Fuerte"
    elif a >= 0.40: return "Moderada"
    elif a >= 0.20: return "Débil"
    else: return "Muy débil"
correlaciones_todas = datos_correlacion.select_dtypes("number").dropna().corr(method="pearson").rename_axis("Variable1").reset_index().melt(id_vars="Variable1", var_name="Variable2", value_name="Correlacion")
correlaciones_todas = correlaciones_todas[correlaciones_todas["Variable1"] != correlaciones_todas["Variable2"]].copy()
correlaciones_todas["Relacion"] = correlaciones_todas["Variable1"] + " vs. " + correlaciones_todas["Variable2"]
correlaciones_todas["Fuerza"] = correlaciones_todas["Correlacion"].apply(fuerza)
correlaciones_todas = correlaciones_todas[["Relacion", "Correlacion", "Fuerza"]].drop_duplicates(subset="Relacion").reset_index(drop=True)
correlaciones_todas.head(10)

```{admonition} Interpretación
:class: tip
Esta tabla usa `.dropna()` sobre las 20 variables a la vez, así que solo conserva los años con dato en **todas**: 2011, 2012, 2013, 2017, 2018 y 2019. Por eso sus valores difieren de la matriz anterior (por ejemplo PM10–PM2.5 da 0,93 aquí y 0,97 en la matriz). Con 6 puntos, una correlación de 0,8 puede aparecer por azar.

Las filas con `Año` miden tendencia: PM10 (−0,96), NO₂ (−0,94) y PM2.5 (−0,87) bajan con el tiempo, mientras que P (0,96) y RGlobal (0,90) suben. En el caso de P es otra vez el error de unidades.
```

In [ ]:
def color_corr(x):
    if x >= 0.80: return "#81C784"
    elif x >= 0.60: return "#A5D6A7"
    elif x >= 0.40: return "#FFF59D"
    elif x >= 0.20: return "#FFE0B2"
    elif x <= -0.80: return "#E57373"
    elif x <= -0.60: return "#EF9A9A"
    elif x <= -0.40: return "#FFCC80"
    elif x <= -0.20: return "#FFE0B2"
    else: return "#E0E0E0"
estilos = pd.DataFrame("", index=correlaciones_todas.index, columns=correlaciones_todas.columns)
estilos["Correlacion"] = ["background-color: " + color_corr(v) + "; color: black" for v in correlaciones_todas["Correlacion"]]
estilos["Fuerza"] = estilos["Correlacion"]
correlaciones_todas.round(2).rename(columns={"Relacion": "Relación", "Correlacion": "Coeficiente de correlación"}).style.apply(lambda _: estilos.rename(columns={"Relacion": "Relación", "Correlacion": "Coeficiente de correlación"}), axis=None).hide(axis="index").set_table_styles([{"selector": "th, td", "props": [("border", "1px solid #999999"), ("padding", "4px 10px"), ("text-align", "left")]}])

### 1.9 Análisis de valores atípicos

*Objetivo específico 1 · valores atípicos.*

In [ ]:
outliers_pm = pd.DataFrame({"Variable": ["PM10", "PM2.5"], "Q1": [pm10["Promedio"].quantile(0.25), pm25["Promedio"].quantile(0.25)], "Q3": [pm10["Promedio"].quantile(0.75), pm25["Promedio"].quantile(0.75)]})
outliers_pm["Rango_IQR"] = outliers_pm["Q3"] - outliers_pm["Q1"]
outliers_pm["Limite_inferior"] = outliers_pm["Q1"] - 1.5 * outliers_pm["Rango_IQR"]
outliers_pm["Limite_superior"] = outliers_pm["Q3"] + 1.5 * outliers_pm["Rango_IQR"]
tabla(outliers_pm, ["Variable", "Primer cuartil", "Tercer cuartil", "Rango IQR", "Límite inferior", "Límite superior"])

```{admonition} Interpretación
:class: tip
Para PM10 son atípicos los valores por encima de 71,5 µg/m³, y para PM2.5 los que superan 30,75 µg/m³. Los límites inferiores son prácticamente cero, así que no hay atípicos bajos.
```

#### Valores atípicos de PM10

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.boxplot(pm10["Promedio"].dropna(), widths=0.4)
ax.set_title("Distribución y valores atípicos de PM10", fontweight="bold"); ax.set_ylabel("Concentración de PM10"); ax.set_xticks([])
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
PM10 tiene 91 registros atípicos (3 %), todos por encima de la caja. Los más altos están en Bogotá (estación Bolivia, 162 µg/m³ en 2012) y en zonas industriales como Yumbo y Ráquira.
```

#### Valores atípicos de PM2.5

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.boxplot(pm25["Promedio"].dropna(), widths=0.4)
ax.set_title("Distribución y valores atípicos de PM2.5", fontweight="bold"); ax.set_ylabel("Concentración de PM2.5"); ax.set_xticks([])
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
PM2.5 tiene 74 atípicos (3,7 %), también solo valores altos. Que se concentren en pocas estaciones apunta a fuentes locales y no a un problema generalizado.
```

#### Valores atípicos de variables meteorológicas

In [ ]:
meteorologicas = df[df["Variable"].isin(["TAire", "HAire", "VViento", "P", "PLiquida", "RGlobal", "RUVb"])]
outliers_meteorologicas = meteorologicas.groupby("Variable", as_index=False).agg(Q1=("Promedio", lambda s: s.quantile(0.25)), Q3=("Promedio", lambda s: s.quantile(0.75)))
outliers_meteorologicas["IQR"] = outliers_meteorologicas["Q3"] - outliers_meteorologicas["Q1"]
outliers_meteorologicas["Limite_inferior"] = outliers_meteorologicas["Q1"] - 1.5 * outliers_meteorologicas["IQR"]
outliers_meteorologicas["Limite_superior"] = outliers_meteorologicas["Q3"] + 1.5 * outliers_meteorologicas["IQR"]
tabla(outliers_meteorologicas, ["Variable", "Primer cuartil", "Tercer cuartil", "Rango IQR", "Límite inferior", "Límite superior"])

In [ ]:
orden_met = sorted(meteorologicas["Variable"].unique())
fig, ax = plt.subplots()
ax.boxplot([meteorologicas.loc[meteorologicas["Variable"] == v, "Promedio"].dropna() for v in orden_met])
ax.set_xticks(range(1, len(orden_met) + 1))
ax.set_xticklabels(orden_met)
ax.set_title("Distribución y valores atípicos de las variables meteorológicas", fontweight="bold"); ax.set_xlabel("Variable meteorológica"); ax.set_ylabel("Valor registrado")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
La presión tiene atípicos altos, que son los registros en hPa. La precipitación tiene valores extremos muy lejanos de la caja (acumulados). El viento tiene pocos atípicos.

**Los atípicos no se eliminan:** pueden ser episodios reales de contaminación. Las excepciones son los errores demostrables (presión en hPa, representatividad mayor a 100 %), que deben corregirse en la fuente.
```

### 1.10 Resumen estadístico general

In [ ]:
variables_resumen = df[df["Variable"].isin(["PM10", "PM2.5", "SO2", "NO2", "O3", "TAire", "HAire", "VViento", "P", "PLiquida", "RGlobal", "RUVb"])].groupby("Variable", as_index=False).agg(N=("Promedio", "count"), Mínimo=("Promedio", "min"), Q1=("Promedio", lambda s: s.quantile(0.25)), Mediana=("Promedio", "median"), Media=("Promedio", "mean"), Q3=("Promedio", lambda s: s.quantile(0.75)), Máximo=("Promedio", "max"))
tabla(variables_resumen, ["Variable", "N", "Mínimo", "Q1", "Mediana", "Media", "Q3", "Máximo"])

```{admonition} Interpretación
:class: tip
Esta tabla resume las variables principales en un solo lugar y cierra el análisis descriptivo.
```

### 1.11 Variable objetivo: excedencia diaria de PM2.5

*Objetivo específico 2: caracterizar la variable que predecirá el modelo.*

El modelo del proyecto responderá: **¿una estación registrará en el año al menos un día con PM2.5 por encima del límite diario de la Resolución 2254 de 2017?** Antes de definirla hay que verificar qué mide exactamente la columna `Excedencias limite actual` en cada tiempo de exposición.

In [ ]:
pm25["Excede"] = pm25["Excedencias limite actual"] > 0
verificacion = pm25.groupby("Tiempo de exposición (horas)").agg(
    Registros=("Excede", "size"),
    Porcentaje_que_excede=("Excede", lambda s: s.mean() * 100),
    Maximo_sin_excedencia=("Máximo", lambda s: s[~pm25.loc[s.index, "Excede"]].max()),
).reset_index()
tabla(verificacion, ["Exposición (h)", "Registros", "% que excede", "Máximo sin excedencia (µg/m³)"], decimales=1)

```{admonition} Interpretación
:class: tip
En los registros de **24 horas** ninguna estación sin excedencias supera 50,2 µg/m³, lo que coincide con el límite diario de la norma. En los de **1 hora**, en cambio, hay estaciones con máximos de 3.608 µg/m³ que figuran sin excedencias: la norma no define un límite horario para PM2.5 y esa columna no es confiable en esos registros.

**Decisión:** la variable objetivo se construye solo con los registros de 24 horas.
```

In [ ]:
pm25_24h = pm25[pm25["Tiempo de exposición (horas)"] == 24].copy()
pm25_24h["Excede"] = pm25_24h["Excede"].astype(int)
print("Registros:", len(pm25_24h), "| Estaciones:", pm25_24h["Estación"].nunique())
balance = pm25_24h["Excede"].value_counts().rename_axis("Excede").reset_index(name="Registros")
balance["Porcentaje"] = balance["Registros"] / balance["Registros"].sum() * 100
tabla(balance, ["Excede (1 = sí)", "Registros", "Porcentaje"], decimales=1)

```{admonition} Interpretación
:class: tip
La variable objetivo tiene **1.144 registros de 383 estaciones**, y el 47,9 % excede el límite. Las clases están **balanceadas**, así que el modelo no necesitará técnicas de sobremuestreo y la exactitud será una métrica razonable (junto con el AUC y el recall).
```

In [ ]:
fig, ax = plt.subplots()
for valor, color, etiqueta in [(0, "#9e9e9e", "No excede"), (1, "#c62828", "Excede")]:
    d = pm25_24h[pm25_24h["Excede"] == valor]
    ax.scatter(d["Año"] + np.random.uniform(-0.25, 0.25, len(d)), d["Máximo"], s=12, alpha=0.6, color=color, label=etiqueta)
ax.step([2010.5, 2018.5, 2024.5], [50, 37, 37], where="post", color="black", linewidth=1.5, label="Límite diario (Res. 2254/2017)")
ax.set_ylim(0, 150); ax.set_xticks(range(2011, 2025))
ax.set_title("Máximo diario de PM2.5 y límite de la norma", fontweight="bold")
ax.set_xlabel("Año"); ax.set_ylabel("Máximo diario de PM2.5 (µg/m³)")
plt.xticks(rotation=45, ha="right"); ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
El gráfico confirma la definición: los registros que exceden (rojo) están por encima de la línea del límite y los que no exceden (gris) por debajo. El límite bajó de **50 a 37 µg/m³** a mediados de 2018, cuando entró en vigor el nivel más estricto de la norma. Por eso una misma concentración puede ser excedencia después de 2018 y no antes.
```

In [ ]:
por_anio = pm25_24h.groupby("Año").agg(Tasa=("Excede", lambda s: s.mean() * 100), Registros=("Excede", "size")).reset_index()
fig, ax = plt.subplots()
ax.bar(por_anio["Año"].astype(str), por_anio["Tasa"], color="#595959")
for x, t, n in zip(por_anio["Año"].astype(str), por_anio["Tasa"], por_anio["Registros"]):
    ax.text(x, t + 1.5, f"{t:.0f}%\n(n={n})", ha="center", fontsize=8)
ax.set_ylim(0, 100); ax.set_title("Porcentaje de estaciones que exceden el límite por año", fontweight="bold")
ax.set_xlabel("Año"); ax.set_ylabel("Estaciones que exceden (%)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
La tasa de excedencia no tiene una tendencia clara: oscila entre 21 % (2011) y 69 % (2020). Los picos de 2016, 2019–2020 y 2024 sugieren años con episodios de contaminación más frecuentes, que podrían relacionarse con condiciones meteorológicas o incendios; el dataset no permite confirmarlo.

Dos efectos complican la lectura: entre 2011 y 2016 hay pocas estaciones (14 a 38), y en 2018 el límite se volvió más estricto. A pesar de que el promedio de PM2.5 bajó (sección 1.6), **los días con excedencias siguen siendo frecuentes**: mejorar el promedio no elimina los episodios críticos.
```

In [ ]:
por_tipo = pm25_24h.groupby("Tipo de Estación").agg(Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
tabla(por_tipo, ["Tipo de estación", "% que excede", "Registros"], decimales=1)

```{admonition} Interpretación
:class: tip
Las estaciones **fijas** exceden con mucha más frecuencia (52 %) que las **indicativas** (20 %). Las fijas miden todo el año y suelen estar en zonas urbanas críticas; las indicativas son campañas cortas, con menos días para registrar una excedencia.
```

In [ ]:
por_depto = pm25_24h.groupby("Departamento_norm").agg(Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
por_depto = por_depto[por_depto["n"] >= 20]
barras_h(por_depto, "Departamento_norm", "Tasa", "Porcentaje de registros que exceden el límite de PM2.5",
         "Registros que exceden (%)", "Departamento", subtitulo="Departamentos con al menos 20 registros", alto=6, dec=0)

```{admonition} Interpretación
:class: tip
Hay un contraste geográfico claro. **Bogotá (65 %), Antioquia (57 %) y Cundinamarca (55 %)** exceden con frecuencia, mientras que **Córdoba (5 %), Magdalena (15 %) y Bolívar (22 %)** casi no lo hacen.

Una explicación física plausible: las ciudades andinas están en valles o altiplanos donde las inversiones térmicas y el poco viento atrapan los contaminantes, mientras que en la costa Caribe los vientos alisios favorecen la dispersión. La ubicación será uno de los predictores más importantes del modelo.
```

In [ ]:
estaciones = pm25_24h.groupby("Estación").agg(Latitud=("Latitud", "mean"), Longitud=("Longitud", "mean"),
                                            Tasa=("Excede", "mean"), n=("Excede", "size")).reset_index()
fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(estaciones["Longitud"], estaciones["Latitud"], c=estaciones["Tasa"] * 100, cmap="Reds",
                s=15 + estaciones["n"] * 4, edgecolor="#555555", linewidth=0.4, vmin=0, vmax=100)
plt.colorbar(sc, ax=ax, shrink=0.7, label="Años con excedencia (%)")
ax.set_title("Estaciones de PM2.5 según frecuencia de excedencia", fontweight="bold")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); ax.set_aspect("equal"); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
El mapa muestra el mismo patrón: los puntos rojos intensos se concentran en el centro andino (Bogotá, Valle de Aburrá) y los claros en el norte (Caribe) y el suroccidente. El tamaño del punto indica cuántos años reportó la estación.
```

In [ ]:
pm25_24h["Cuartil_datos"] = pd.qcut(pm25_24h["No. de datos"], 4, labels=["Q1 (menos días)", "Q2", "Q3", "Q4 (más días)"])
esfuerzo = pm25_24h.groupby("Cuartil_datos", observed=True).agg(
    Dias_medidos=("No. de datos", "median"), Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
tabla(esfuerzo, ["Cuartil de días medidos", "Mediana de días medidos", "% que excede", "Registros"], decimales=1)

```{admonition} Interpretación
:class: tip
Este es el hallazgo más importante para el modelo. Las estaciones que midieron pocos días (Q1, mediana de 42 días) exceden en el 21 % de los casos, y las que midieron casi todo el año (Q4) en el 79 %. **Mientras más días se mide, más probable es captar al menos un día malo.**
```

```{admonition} Precaución
:class: warning
La variable objetivo depende del **esfuerzo de medición** y no solo de la calidad del aire. El modelo debe incluir `No. de datos` (o la representatividad) como predictor de control. Si no, confundirá «estación que mide mucho» con «estación contaminada».
```

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.boxplot([pm25_24h.loc[pm25_24h["Excede"] == 0, "Promedio"], pm25_24h.loc[pm25_24h["Excede"] == 1, "Promedio"]],
           widths=0.5)
ax.set_xticks([1, 2], ["No excede", "Excede"])
ax.set_title("Promedio anual de PM2.5 según excedencia", fontweight="bold")
ax.set_ylabel("Promedio anual de PM2.5 (µg/m³)"); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Las estaciones que exceden tienen un promedio anual mayor (mediana de 18,4 frente a 13,1 µg/m³), pero las cajas se solapan: hay estaciones con promedio bajo que igual tienen días críticos. El promedio anual por sí solo no basta para anticipar las excedencias.
```

```{admonition} Precaución
:class: warning
`Promedio`, `Mediana`, `Percentil 98`, `Máximo`, `Días de excedencias` y `Porcentaje excedencias limite actual` de PM2.5 se calculan con las mismas mediciones que definen la excedencia. Usarlas como predictores sería **fuga de información** (*data leakage*): el modelo acertaría en el EDA pero sería inútil en la práctica.
```

#### Predictores candidatos para el modelo

| Variable | Uso | Razón |
|---|---|---|
| Latitud, Longitud, Departamento | ✅ Incluir | Fuerte contraste andino–Caribe |
| Tipo de estación | ✅ Incluir | Fijas 52 % vs. indicativas 20 % |
| No. de datos, Representatividad | ✅ Incluir (control) | Controlan el esfuerzo de medición |
| Año | ✅ Incluir | Captura el cambio de límite (2018) y la evolución de la red |
| Autoridad ambiental | ⚠️ Evaluar | Puede repetir la información de ubicación |
| Promedio, Mediana, P98, Máximo, Días de excedencias de PM2.5 | ❌ Excluir | Fuga de información |
| Meteorología de la misma estación | ⚠️ Evaluar | Aporta física, pero solo ~30 % de las estaciones la mide |

```{admonition} Interpretación
:class: tip
Con estos predictores, una prueba rápida con un bosque aleatorio y validación por estación (las estaciones de entrenamiento y de prueba son distintas) alcanzó un **AUC de 0,81**. El problema es viable; la etapa de modelado deberá comparar algoritmos, ajustar hiperparámetros y evaluar qué aporta la meteorología.
```